In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = "retina"
import numpy as np
import contextily as cx
import os
from matplotlib import pyplot as plot
import shapely
from shapely import wkt

In [ ]:
# first ensure you are in the correct directory!
# os.chdir("/users/username/MinGenResources-DFW/Parks")

In [ ]:
# read in isochrone data
isochrones = gpd.read_file("data/holes_no_parks.geojson")

# read in geocoded census data
age = gpd.read_file("data/demographic_data/geo_age.geojson")
income = gpd.read_file("data/demographic_data/geo_income.geojson")
poverty = gpd.read_file("data/demographic_data/geo_poverty.geojson")

In [ ]:
# change to crs used for mapping
isochrones = isochrones.to_crs("EPSG:3857")

In [ ]:
# In the data frame "geometry" is the covered areas and "holes" are the uncovered areas
# separate them into two data frames for coverage and holes
coverage = isochrones[["eps", "geometry"]]
holes = isochrones[["eps", "holes"]]

In [ ]:
# Uses wkt to change string to geometry
holes["holes"] = holes["holes"].apply(lambda x: shapely.wkt.loads(x))
# assign the holes column as the geometry and to the CRS it was originallly assigned
holes = holes.set_geometry("holes", crs = "EPSG:3083")
# change to crs used for mapping
holes = holes.to_crs("EPSG:3857")

In [ ]:
# Since the coverage polygons have the hole areas blank, and the hole polygons have the coverage areas blank, we want to flip them
# so it is more intuitive when combining with the census data. This way, when we plot the coverage data, we will see the
# census data corresponding to the coverage areas since they are blank, and the same for the holes
holes_temp = coverage
coverage = holes
holes = holes_temp

In [ ]:
# plot to check that it is correct
coverage[coverage["eps"] == 20].plot()
holes[holes["eps"] == 20].plot()

In [ ]:
# Here a function is defined to make plotting the maps easier
# it takes in the data frame to plot (demographic data), the column of the data frame to plot
# the desired title of the plot and the isochrone data that we want to use (holes or coverage)
def plot_data(data_frame, column_name, title, isochrones):
    # plots the data without any isochrones
    ax = data_frame.plot(column = column_name, legend = True)
    ax.set_title(title)
    ax.set_axis_off();
    # plot for each "epsilon" value of 10, 15, and 20 minutes of walking distance to a park
    # plots the isochrones overlaid on the census data
    for i in range(0, isochrones.shape[0]):
        ax = data_frame.plot(column = column_name, legend = True)
        isochrones[isochrones["eps"] == isochrones.loc[i].iloc[0]].plot(ax = ax, color = "white", alpha = 0.9)
        ax.set_title("{title}: {distance} minute walk".format(title = title, distance = isochrones.loc[i].iloc[0] ))
        ax.set_axis_off();

#### Mapping Age Data

In [ ]:
# investigating the highest percentage value
#age["percent_child"].sort_values(ascending = False)
#age.loc[563]

In [ ]:
# percent of population below 18
plot_data(age, "percent_child", "Percent of Population Below 18", holes)

In [ ]:
# absolute number of children in each block group
plot_data(age, "Children", "Number of Children by Block Group", holes)

#### Mapping Income Data

In [ ]:
# median income
plot_data(income, "Median_Income", "Median Income of Block Group", holes)

#### Mapping Poverty Data

In [ ]:
# households below the poverty level
plot_data(poverty, "households_below_poverty", "Number of Households below the Poverty Level", holes)

In [ ]:
# percent of households below the poverty level
plot_data(poverty, "percent_below", "Percent of Households below the Poverty Level", holes)

In [ ]:
# households with children below the poverty level
plot_data(poverty, "households_with_children", "Households with Children below the Poverty Level", holes)

In [ ]:
# percent of households in each block group who have children and are below the poverty level
plot_data(poverty, "child_poverty_percent", "Percent of Households with Children below the Poverty Level", holes)